In [3]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Suhitha\Downloads\ML Projects\MY_ALL_PROJECTS\NLP\spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [4]:
df=df[['v1','v2']]
df.columns=["label","message"]
print(df.shape)
print(df.isnull().sum())
print(df['label'].value_counts())

(5572, 2)
label      0
message    0
dtype: int64
label
ham     4825
spam     747
Name: count, dtype: int64


In [5]:
print(df.columns.tolist())

['label', 'message']


In [6]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [12]:
!pip install nltk

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 9.3 MB/s  0:00:00


In [7]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Suhitha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Suhitha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [10]:
import string
from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))
def clean_text(text):
    text=text.lower()
    text=text.translate(str.maketrans(' ',' ',string.punctuation))
    words=text.split()
    words=[word for word in words if word not in stop_words]
    return' '.join(words)

df['clean_message'] = df['message'].apply(clean_text)
df[['message','clean_message']].head()


,message,clean_message
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though


#### TF-IDF Vectorization

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=3000)
x= tfidf.fit_transform(df['clean_message']).toarray()
y=df['label'].map({'ham':0,'spam':1})
print(X.shape)
print(y.value_counts())

(5572, 3000)
label
0    4825
1     747
Name: count, dtype: int64


##### Train/Test split+Navive Bayes Model

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
model=MultinomialNB()
model.fit(X_train,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


##### Evaluate the Model

In [18]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
y_pred=model.predict(X_test)
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.97847533632287
[[965   0]
 [ 24 126]]
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       965
           1       1.00      0.84      0.91       150

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [19]:
print(confusion_matrix(y_test,y_pred))

[[965   0]
 [ 24 126]]


In [20]:
import pickle
pickle.dump(model,open('spam_model.pkl','wb'))
pickle.dump(tfidf,open('vectorizer.pkl','wb'))